# Strategy: `rf_vw_gr_top_n_25`

Documents the live scoring pipeline for the production fundamentals-alpha strategy.
Each section runs one stage and explains the design choice.

| Component | Meaning |
|-----------|--------|
| `gr` | **Composite score** — cross-sectional z-score of value + quality + momentum factors |
| `vw` | **Volatility-weighted** — inverse-vol position sizing, capped at 10% per stock |
| `rf` | **Regime filter** — SPY 12-month return cuts exposure to 50% in extreme markets |
| `top_n_25` | Select the top 25 stocks after all filters and guardrails |

**Backtest (2005–2025, 211 months):** CAGR 21.6% · Sharpe 1.37 · Max DD -21.3% · Beta 0.86 · Win rate 71.1%

**Entry point:** `scripts/score_live.py`  
**Core libraries:** `historic_fundamentals.baselines`, `.universe`, `.risk`

In [1]:
import os
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

# Absolute project root — this notebook lives in <root>/notebooks/
ROOT = Path("/home/pedro/projects/fin_import2")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

from score_live import (
    score_universe,
    _compute_stock_vols, _compute_regime,
    _FACTOR_COLS_FOR_MISSING,
)
from historic_fundamentals.baselines import (
    BASELINE_FACTORS, _VALUE_COLS, _VALUE_SIGN,
    _QUALITY_COLS, _QUALITY_SIGN, _MOMENTUM_COL,
)
from historic_fundamentals.universe import UNIVERSE_DEFAULTS

HF_DB_PATH = str(ROOT / "data" / "historic_fundamentals.duckdb")
AV_DB_PATH = str(ROOT / "data" / "av_financials.duckdb")
PRICES_DB  = "/home/pedro/projects/trade_systems/data/prices.duckdb"

TODAY = date.today()
TOP_N = 25
print(f"ROOT:       {ROOT}")
print(f"HF_DB_PATH: {HF_DB_PATH}")
print(f"AV_DB_PATH: {AV_DB_PATH}")
print(f"Today:      {TODAY}")

ROOT:       /home/pedro/projects/fin_import2
HF_DB_PATH: /home/pedro/projects/fin_import2/data/historic_fundamentals.duckdb
AV_DB_PATH: /home/pedro/projects/fin_import2/data/av_financials.duckdb
Today:      2026-06-04


## Scored universe

`score_universe()` runs the full pipeline up to and including the composite score and guardrails:

1. **Load features** — most recent row per ticker where `feature_available_date ≤ today` (PIT-safe)
2. **Join metadata** — sector, company name from `company_overview`
3. **Universe filters** — market cap ≥ $1B · price ≥ $5 · exclude REIT + financials · ADDV ≥ $5M
4. **Composite score** — cross-sectional z-score of value + quality + momentum
5. **Guardrails** — flag value traps; tag missing-factor count

Portfolio construction (top-N selection, volatility weighting, regime filter) happens after.

In [2]:
scored = score_universe(
    hf_db_path=HF_DB_PATH,
    av_db_path=AV_DB_PATH,
    model_path=None,
    today=TODAY,
)

print(f"Scored universe: {len(scored):,} tickers")
print(f"Score range: {scored['score'].min():.3f} to {scored['score'].max():.3f}")
print(f"Value traps flagged: {scored['value_trap'].sum()}")
print()
print("Sector breakdown:")
print(scored['sector'].value_counts().to_string())

Scored universe: 1,425 tickers
Score range: -3.818 to 2.794
Value traps flagged: 15

Sector breakdown:
sector
INDUSTRIALS               279
TECHNOLOGY                278
HEALTHCARE                269
CONSUMER CYCLICAL         203
BASIC MATERIALS           105
ENERGY                    102
CONSUMER DEFENSIVE         81
UTILITIES                  57
COMMUNICATION SERVICES     51


## Step 1 — Point-in-time safety

Each row in `monthly_pe` stores a `feature_available_date` — the earliest date the underlying
fundamentals were assumed to be publicly available (quarterly filing + 60-day lag; annual + 90 days).

At score time, only rows where `feature_available_date ≤ today` are loaded, so the model never
sees earnings that haven't been filed yet.

In [3]:
import duckdb
conn = duckdb.connect(HF_DB_PATH, read_only=True)
pit_check = conn.execute("""
    SELECT
        COUNT(DISTINCT ticker) AS total_tickers,
        SUM(CASE WHEN feature_available_date <= CURRENT_DATE THEN 1 ELSE 0 END) AS pit_available,
        SUM(CASE WHEN feature_available_date >  CURRENT_DATE THEN 1 ELSE 0 END) AS pit_blocked
    FROM (
        SELECT ticker, MAX(month_end_date) AS latest_month,
               MAX(feature_available_date) AS feature_available_date
        FROM monthly_pe
        GROUP BY ticker
    )
""").df()
conn.close()

print("PIT status for latest row per ticker:")
print(pit_check.to_string(index=False))
print()
print("feature_available_date distribution in scored universe (top 5 dates):")
print(pd.to_datetime(scored['feature_available_date']).dt.date.value_counts().head(5).to_string())

PIT status for latest row per ticker:
 total_tickers  pit_available  pit_blocked
          2653         2529.0        103.0

feature_available_date distribution in scored universe (top 5 dates):
feature_available_date
2026-05-30    1243
2026-05-01      58
2026-04-01      49
2026-04-29      43
2026-03-01       6


## Step 2 — Universe filters

Five sequential filters define the investable universe:

| Filter | Threshold | Rationale |
|--------|-----------|----------|
| Market cap | ≥ $1B | Micro/small caps have high market impact cost |
| Price | ≥ $5 | Excludes penny stocks |
| Sector required | Yes | Sector-relative scoring needs a valid sector label |
| Excluded sectors | REAL ESTATE, FINANCIAL SERVICES | Non-comparable accounting; different valuation regime |
| Avg daily $ volume | ≥ $5M (30-day) | Minimum liquidity to trade 25 positions at size |

In [4]:
print("UNIVERSE_DEFAULTS:")
for k, v in UNIVERSE_DEFAULTS.items():
    print(f"  {k:<30} {v}")

UNIVERSE_DEFAULTS:
  min_market_cap                 1000000000
  min_price                      5.0
  require_sector                 True
  excluded_sectors               ['REAL ESTATE', 'FINANCIAL SERVICES']
  min_avg_dollar_volume          5000000


## Step 3 — Composite score (`gr`)

The `gr` score is a **cross-sectional average of z-scored factor signals** across three groups.

**Value** — what are you paying?

| Factor | Direction |
|--------|-----------|
| `ps_ratio` | lower = better |
| `fcf_yield` | higher = better |
| `ev_ebitda` | lower = better |
| `earnings_yield` | higher = better |

**Quality** — is the business good?

| Factor | Direction |
|--------|-----------|
| `roic` | higher = better |
| `roa` | higher = better |
| `operating_margin_slope_5y` | higher = better |
| `earnings_quality` | higher = better |
| `asset_growth` | lower = better (overinvestment predicts underperformance) |

**Momentum**

| Factor | Direction |
|--------|-----------|
| `momentum_12_1` | higher = better |

For each stock in the live cross-section each factor is z-scored, sign-flipped where
lower is better, then averaged. Missing factors contribute 0 (neutral); the average is
normalised by the count of present factors so sparse data doesn't inflate scores.

In [5]:
all_factor_cols = _VALUE_COLS + _QUALITY_COLS + [_MOMENTUM_COL]

print("Factor coverage in the scored universe:")
for col in all_factor_cols:
    if col in scored.columns:
        cov  = scored[col].notna().mean()
        grp  = "value" if col in _VALUE_COLS else ("quality" if col in _QUALITY_COLS else "momentum")
        sign = "lower=better" if {**_VALUE_SIGN, **_QUALITY_SIGN}.get(col, False) else "higher=better"
        print(f"  {col:<35} {cov:.0%}  [{grp} / {sign}]")

Factor coverage in the scored universe:
  ps_ratio                            98%  [value / lower=better]
  fcf_yield                           99%  [value / higher=better]
  ev_ebitda                           92%  [value / lower=better]
  earnings_yield                      99%  [value / higher=better]
  roic                                98%  [quality / higher=better]
  roa                                 99%  [quality / higher=better]
  operating_margin_slope_5y           94%  [quality / higher=better]
  earnings_quality                    99%  [quality / higher=better]
  asset_growth                        99%  [quality / lower=better]
  momentum_12_1                       99%  [momentum / higher=better]


In [6]:
ranked = scored.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1   # recompute after sort

show = ["rank", "ticker", "sector", "score", "top_factor",
        "ps_ratio", "fcf_yield", "earnings_yield", "roic", "momentum_12_1"]
show = [c for c in show if c in ranked.columns]

print("Top 10:")
print(ranked.head(10)[show].round(3).to_string(index=False))
print()
print("Bottom 10:")
print(ranked.tail(10)[show].round(3).to_string(index=False))

Top 10:
 rank ticker            sector  score          top_factor  ps_ratio  fcf_yield  earnings_yield   roic  momentum_12_1
    1   SNDK        TECHNOLOGY  2.794      high_fcf_yield    20.439      0.017           0.017 -0.091         43.972
    2   CRNX        HEALTHCARE  2.430 high_earnings_yield   187.873     -0.124          -0.146 -0.368          0.165
    3   WINA CONSUMER CYCLICAL  1.860 high_earnings_yield    16.339      0.030           0.029 60.623         -0.081
    4   BEAM        HEALTHCARE  1.746      high_fcf_yield    18.233     -0.128          -0.022 -0.076          1.080
    5   FTNT        TECHNOLOGY  1.464      high_fcf_yield    15.553      0.022           0.018 69.066          0.356
    6    ZTO       INDUSTRIALS  1.452      high_fcf_yield     0.352      0.793           0.508  0.106          0.297
    7     EE            ENERGY  1.033      high_fcf_yield     0.802      0.194           0.037  0.950          0.171
    8    MAR CONSUMER CYCLICAL  0.837      high_div_yiel

## Step 4 — Guardrails

**Value trap flag**: a stock is flagged if it is in the **top quintile by score** (cheap/high-quality)
*and* shows at least one sign of fundamental deterioration:
- Negative operating margin
- Debt/EBITDA > 10×
- ROA < −5%
- Earnings yield < −5%

**Missing data**: stocks with more than 2 missing factors are excluded — the composite score
is unreliable when most inputs are absent.

In [7]:
n_traps = scored["value_trap"].sum()
n_poor  = (scored["missing_factor_count"] > 2).sum()
print(f"Value traps: {n_traps}  ({n_traps/len(scored):.1%} of universe)")
print(f"Poor data quality (>2 missing factors): {n_poor}")
print()

if n_traps > 0:
    trap_cols = ["ticker", "sector", "score",
                 "ttm_operating_margin", "debt_to_ebitda", "roa", "earnings_yield"]
    tc = [c for c in trap_cols if c in scored.columns]
    print("Value trap examples (top 8):")
    print(scored[scored["value_trap"]].sort_values("score", ascending=False)[tc].head(8).round(3).to_string(index=False))

print()
print("Data quality breakdown:")
print(scored["data_quality"].value_counts().to_string())

Value traps: 15  (1.1% of universe)
Poor data quality (>2 missing factors): 98

Value trap examples (top 8):
ticker      sector  score  ttm_operating_margin  debt_to_ebitda    roa  earnings_yield
  CRNX  HEALTHCARE  2.430               -30.203             NaN -0.354          -0.146
  BEAM  HEALTHCARE  1.746                -2.266             NaN -0.044          -0.022
  VSAT  TECHNOLOGY  0.508                -0.010           0.088 -0.023          -0.035
   MXL  TECHNOLOGY  0.334                -0.159           0.959 -0.171          -0.017
  AEHR  TECHNOLOGY  0.323                -0.343           0.257 -0.073          -0.003
   NSP INDUSTRIALS  0.322                -0.002           0.528 -0.011          -0.018
  LITE  TECHNOLOGY  0.296                 0.095          12.067  0.063           0.004
  SEPN  HEALTHCARE  0.285                -0.771             NaN -0.063          -0.027

Data quality breakdown:
data_quality
good       852
partial    475
poor        98


In [8]:
clean = scored[
    (~scored["value_trap"]) &
    (scored["missing_factor_count"] <= 2)
].copy()
print(f"After guardrails: {len(scored):,} → {len(clean):,} tickers  ({len(scored)-len(clean)} removed)")

After guardrails: 1,425 → 1,315 tickers  (110 removed)


## Step 5 — Regime filter (`rf`)

Portfolio exposure is modulated by the SPY 12-month return:

| SPY 12m return | Regime | Exposure |
|----------------|--------|----------|
| > +25% | Overbought | 50% invested, 50% cash |
| < −20% | Oversold / bear | 50% invested, 50% cash |
| −20% to +25% | Neutral | 100% invested |

Reducing exposure symmetrically in both extremes shrank max drawdown from −26% → −21%
in the backtest at a cost of roughly 3% CAGR.

In [9]:
regime_exposure, spy_r12, regime_label = _compute_regime(PRICES_DB)

print(f"SPY 12-month return: {spy_r12:+.1%}")
print(f"Regime:             {regime_label}")
print(f"Exposure:           {regime_exposure:.0%}")
if regime_exposure < 1.0:
    cash_pct = (1 - regime_exposure) * 100
    print(f"=> Positions cut to {regime_exposure:.0%}; {cash_pct:.0f}% held as cash")

SPY 12-month return: +23.1%
Regime:             FULL 100% (SPY 12m=+23.1%)
Exposure:           100%


## Step 6 — Volatility weighting (`vw`)

Positions are sized by **inverse trailing 12-month volatility** so that each stock contributes
roughly equal risk rather than equal capital.

**Algorithm:**
1. Compute 12-month trailing monthly-return std dev for each top-N ticker
2. `weight ∝ 1 / vol`, normalised to 100%
3. Cap at 10% per position; redistribute excess iteratively (20 passes)
4. `alloc_pct = weight_pct × regime_exposure`

Effect: high-beta names naturally receive smaller weights than low-beta names.

In [10]:
# Select top-N with 25% sector cap
MAX_SECTOR_PCT = 0.25
clean_ranked   = clean.sort_values("score", ascending=False).reset_index(drop=True)

sector_counts = {}
top_rows = []
for _, row in clean_ranked.iterrows():
    sec = row.get("sector", "")
    cap = max(1, int(TOP_N * MAX_SECTOR_PCT))
    if sector_counts.get(sec, 0) >= cap:
        continue
    top_rows.append(row)
    sector_counts[sec] = sector_counts.get(sec, 0) + 1
    if len(top_rows) == TOP_N:
        break

top_df      = pd.DataFrame(top_rows).reset_index(drop=True)
top_tickers = top_df["ticker"].tolist()

print(f"Top-{TOP_N} selected")
print(f"Sector distribution: {dict(top_df['sector'].value_counts())}")

Top-25 selected
Sector distribution: {'TECHNOLOGY': np.int64(6), 'CONSUMER CYCLICAL': np.int64(6), 'HEALTHCARE': np.int64(4), 'ENERGY': np.int64(3), 'COMMUNICATION SERVICES': np.int64(3), 'CONSUMER DEFENSIVE': np.int64(2), 'INDUSTRIALS': np.int64(1)}


In [11]:
vols = _compute_stock_vols(PRICES_DB, top_tickers, lookback_months=12)

print("12-month trailing vol (monthly std dev):")
print(f"  min {vols.min():.3f}  median {vols.median():.3f}  max {vols.max():.3f}")
print(f"  Coverage: {vols.notna().sum()} / {len(top_tickers)} tickers")

inv_vol = 1.0 / vols.reindex(top_tickers).fillna(vols.median())
weights = inv_vol / inv_vol.sum() * 100.0

CAP = 10.0
for _ in range(20):
    excess   = (weights - CAP).clip(lower=0).sum()
    weights  = weights.clip(upper=CAP)
    uncapped = weights[weights < CAP]
    if excess < 0.01 or uncapped.empty:
        break
    weights[weights < CAP] += excess * (uncapped / uncapped.sum())
weights = weights / weights.sum() * 100.0

top_df["weight_pct"] = top_df["ticker"].map(weights).round(2)
top_df["alloc_pct"]  = (top_df["weight_pct"] * regime_exposure).round(2)

print(f"\nWeight range: {weights.min():.1f}% – {weights.max():.1f}%")
print(f"Total deployed: {top_df['alloc_pct'].sum():.1f}%   Cash: {100-top_df['alloc_pct'].sum():.1f}%")

12-month trailing vol (monthly std dev):
  min 0.063  median 0.121  max 0.490
  Coverage: 25 / 25 tickers

Weight range: 0.9% – 7.1%
Total deployed: 100.0%   Cash: -0.0%


## Final portfolio

Complete ranked portfolio with allocations.

In [12]:
top_df = top_df.copy()
top_df["rank"] = range(1, len(top_df) + 1)

pct_cols = ["fcf_yield", "earnings_yield", "roa", "ttm_gross_margin", "ttm_operating_margin"]
display  = top_df.copy()
for c in pct_cols:
    if c in display.columns:
        display[c] = (display[c] * 100).round(1).astype(str) + "%"

show_cols = [
    "rank", "ticker", "sector", "score", "top_factor",
    "weight_pct", "alloc_pct",
    "ps_ratio", "fcf_yield", "earnings_yield",
    "ttm_gross_margin", "ttm_operating_margin",
    "roa", "debt_to_ebitda",
    "data_quality", "feature_available_date",
]
show = [c for c in show_cols if c in display.columns]
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
print(display[show].to_string(index=False))

 rank ticker                 sector    score          top_factor  weight_pct  alloc_pct  ps_ratio fcf_yield earnings_yield ttm_gross_margin ttm_operating_margin    roa  debt_to_ebitda data_quality feature_available_date
    1   SNDK             TECHNOLOGY 2.793944      high_fcf_yield        0.91       0.91 20.439056      1.7%           1.7%            56.0%                40.9%  26.4%        0.007258      partial             2026-05-30
    2   WINA      CONSUMER CYCLICAL 1.860172 high_earnings_yield        4.18       4.18 16.338641      3.0%           2.9%            96.7%                62.8% 116.8%        0.011405         good             2026-05-30
    3   FTNT             TECHNOLOGY 1.463684      high_fcf_yield        2.22       2.22 15.552881      2.2%           1.8%            80.7%                31.1%  19.8%        0.383397      partial             2026-05-30
    4    ZTO            INDUSTRIALS 1.451923      high_fcf_yield        5.86       5.86  0.352050     79.3%          50.

## Pipeline summary

```
monthly_pe (historic_fundamentals.duckdb)
    │
    ▼
[1] Load features  — 1 row per ticker, feature_available_date ≤ today
[2] Universe filters
    │  mkt cap ≥$1B · price ≥$5 · sector required
    │  exclude REIT + Financials · ADDV ≥$5M
    ▼  ~800–1,000 tickers
[3] Composite score (gr)
    │  cross-sectional z-score:
    │    value:    ps_ratio, fcf_yield, ev_ebitda, earnings_yield
    │    quality:  roic, roa, margin_slope_5y, earnings_quality, asset_growth
    │    momentum: momentum_12_1
    ▼  all tickers ranked within their live cross-section
[4] Guardrails
    │  remove value traps · remove >2 missing factors
    ▼  ~700–900 tickers
[5] Select top-25  — 25% sector cap
    ▼  25 tickers
[6] Volatility weighting (vw)
    │  weight ∝ 1/vol_12m · cap 10% · renormalise
    ▼
[7] Regime filter (rf)
    │  SPY 12m >+25% or <-20%  →  alloc_pct × 0.5
    │  otherwise               →  alloc_pct × 1.0
    ▼
[8] Output  →  docs/live_scores_{YYYYMMDD}_rf_vw_gr_top_n_25.csv
```

**CLI:**
```bash
uv run scripts/score_live.py --top 25
```